In [1]:
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec
from icecream import ic
import jax
import jax.numpy as jnp
from memo import memo
from enum import IntEnum
from ipywidgets import interact, fixed
import ipywidgets as widgets
import pandas as pd

We assume the giver makes a decision acording to 

$$
    U(a) = (E - R(a)) + \lambda R(a) + \delta D_{KL} [ p(\lambda | a ) | \rho(\lambda) ]
$$

where $E \in [E_\text{poor}, E_\text{rich}]$ is the endowment, with $E_\text{poor} = 0.20$ and $E_\text{rich}=1$, and $\lambda$ is a welfare trade-off ratio and actions are in $[0, .25, .5, .75, 1]$. 

In [2]:
# type:ignore

K_values = 5
alpha_idx = jnp.arange(K_values)
alphas = jnp.linspace(0, 1, alpha_idx.size)

beta_idx = jnp.arange(K_values)
betas = jnp.linspace(0, 2, beta_idx.size)

lambda_idx = jnp.arange(K_values)
lambdas = jnp.linspace(0, 2, lambda_idx.size)

delta_idx = jnp.arange(K_values)
deltas = jnp.linspace(0, 1, delta_idx.size)

gamma_idx = jnp.arange(K_values)
gammas = jnp.linspace(0, 10, gamma_idx.size)

class Endowment(IntEnum): 
    Poor=0
    Rich=1

class A(IntEnum):
    Low=0
    Medium=1
    High=2

endowments_vals = jnp.array([.20, 1])

actions = jnp.linspace(0, 1, 5)
actions = jnp.array([0, .15, .25, .5, .9])
actions = jnp.array([0, .10, .18, .25, .5, .9])
print(actions)

#action_payoffs = jnp.array([
#        jnp.linspace(0, endowments_vals[0], len(A)),                   # Poor
#        jnp.linspace(endowments_vals[0]+1, endowments_vals[1], len(A)),  # Rich
#    ])

@jax.jit
def get_endowment_value(endowment):
    return endowments_vals[endowment]

@jax.jit
def get_action_given_endowment(endowment, a, lambda_):
    return endowment - a + lambda_ * a - (a > endowment) * 1e1

@jax.jit
def get_endowment_likelihood(endowment, belief):
    return (1 - endowment) * belief + endowment * (1 - belief)

@jax.jit
def get_prior(param, mean, sd):
    return jax.scipy.stats.norm.pdf(param, mean, sd)


@memo
def Alice_gives_Bob_naive[a:actions, endowment:Endowment, lambda_:lambdas](inv_temp: float, endowment_belief: float):

    Alice: given(lambda_ in lambdas, wpp=1)
    Alice: given(endowment in Endowment, wpp=get_endowment_likelihood(endowment, endowment_belief))
    Alice: chooses (a in actions, wpp=exp(inv_temp * get_action_given_endowment(get_endowment_value(endowment), a, lambda_)))

    return Pr [ Alice.a == a, Alice.endowment == endowment, Alice.lambda_ == lambda_]


@memo
def Alice_gives_Bob[a:actions, endowment:Endowment, lambda_:lambdas, delta:deltas](
                                                   inv_temp: float,
                                                   endowment_belief: float,
                                                   pres_target: float,
                                                   pres_sd: float):

    Alice: given(lambda_ in lambdas, wpp=1)
    Alice: given(delta in deltas, wpp=1)
    Alice: given(endowment in Endowment, wpp=get_endowment_likelihood(endowment, endowment_belief))

    Alice: thinks [
        Bob: thinks [
            Alice: given(lambda_ in lambdas, wpp=1),
            Alice: given(endowment in Endowment, wpp=get_endowment_likelihood(endowment, endowment_belief)),
            Alice: chooses(a in actions, wpp=exp(inv_temp * get_action_given_endowment(get_endowment_value(endowment), a, lambda_))),

            # Presentational goal of A for C
            Alice: given(lambda_pres in lambdas, wpp=get_prior(lambda_pres, pres_target, pres_sd)),
        ],
    ]

    Alice: chooses (a in actions, wpp=exp( inv_temp * (
        get_action_given_endowment(get_endowment_value(endowment), a, lambda_) # Alice wants to give according to her alpha, beta: welfare tradeoff
        + imagine [
            Bob: observes [Alice.a] is a,
            - delta * Bob [ KL [ Alice.lambda_ | Alice.lambda_pres ]] # Alice wants Bob to think Alice's lambda_ is close to get_prior(Alice.lambda_pres, pres_target, pres_sd)
        ]
    )))

    return Pr [ Alice.a == a, Alice.endowment == endowment, Alice.lambda_ == lambda_, Alice.delta == delta]


[0.   0.1  0.18 0.25 0.5  0.9 ]


In [3]:
# Create custom widgets that display both value and name
# Create custom widgets with sufficient width for descriptions
def create_labeled_dropdown(options, description, value=None):
    # Convert options to display both value and name
    labeled_options = [(f"{option.name} ({option.value})", option) for option in options]
    
    # Create dropdown with style to accommodate longer descriptions
    dropdown = widgets.Dropdown(
        options=labeled_options, 
        description=description,
        value=value if value is not None else labeled_options[0][1],
        style={'description_width': 'auto'}  # This allows description to use needed space
    )
    
    # Set layout to provide more space
    dropdown.layout.width = '50%'  # Use full width
    
    return dropdown

# Create your custom widgets
#endowment_widget = widgets.FloatSlider(min=1, max=20, step=1, value=20, description="Alice's true endowment", style={'description_width': 'auto'})
endowment_widget = create_labeled_dropdown(Endowment, "Alice's true endowment", value=Endowment.Rich)
action_inv_temp_widget = widgets.FloatSlider(min=0, max=10, step=.2, value=4, description="Alice's action noise (inv_temp)", style={'description_width': 'auto'})
delta_widget = widgets.FloatSlider(min=0, max=1, step=0.1, value=1, description="Alice's care about what Bob thinks: delta", style={'description_width': 'auto'})
#endowment_poor_widget = widgets.FloatSlider(min=1, max=20, step=1, value=1, description="- Bob's Belief of Alice's endowment (Poor)", style={'description_width': 'auto'})
#endowment_rich_widget = widgets.FloatSlider(min=1, max=20, step=1, value=1, description="- Bob's Belief of Alice's endowment (Rich)", style={'description_width': 'auto'})
endowment_belief_widget = widgets.FloatSlider(min=.01, max=.99, step=0.01, value=0.9, description="Bob's Belief that Alice is Poor", style={'description_width': 'auto'})
#endowment_sd_widget = widgets.FloatSlider(min=0.2, max=5, step=.2, value=2, description="- Bob's Belief SD", style={'description_width': 'auto'})
pres_target_widget = widgets.FloatSlider(min=0, max=2, step=0.1, value=2, description="Alice's target for Bob's inference of her welfare tradeoff ratio - lambda =", style={'description_width': 'auto'})
pres_sd_widget = widgets.FloatSlider(min=0.05, max=1, step=.05, value=0.5, description="- Target SD", style={'description_width': 'auto'})

action_inv_temp_widget.layout.width = '50%'
endowment_widget.layout.width = '50%'
delta_widget.layout.width = '50%'
endowment_belief_widget.layout.width = '50%'
pres_target_widget.layout.width = '50%'
pres_sd_widget.layout.width = '50%'

def show_interactive(
        inv_temp,
        endowment_true,
        delta,
        endowment_belief,
        pres_target,
        pres_sd,
    ):

    alpha = alphas[-1]
    delta_idx = jnp.argmin(jnp.abs(deltas - delta))
    endowment_true_label = Endowment(endowment_true).name
    
    fig, axes = plt.subplots(2, 4, figsize=(14, 8))

    # Joint action params
    p_action_params = Alice_gives_Bob(inv_temp, endowment_belief, pres_target, pres_sd)
    p_action_params /= p_action_params.sum()


    # Unkown endowment, unknown delta
    ax = axes[0, 1]
    ax.set_title('Observer\n$p(\\lambda|a) = \\sum_{E, \\delta} p(\\lambda, E, \\delta | a)$', fontsize=12)
    p_action_params_joint = p_action_params.sum((-3, -1)) / p_action_params.sum((-3, -1)).sum(1, keepdims=True)
    lambdas_means_obs = p_action_params_joint @ lambdas

    sns.heatmap(
        p_action_params_joint.T,
        square=True if K_values == len(actions) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Greens",
        vmin=p_action_params_joint.min(),
        vmax=p_action_params_joint.max(),
        yticklabels=[f'{lambda_:.2f}' for lambda_ in lambdas],
        xticklabels=[f'{a:.2f}' for a in actions],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\lambda|a)$", size=13, rotation=90)
    #ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)

    # Unknown endowment, known delta
    ax = axes[1, 1]
    ax.sharex(axes[0, 0])
    ax.sharey(axes[0, 0])

    ax.set_title(f'Observer knows $\\delta$\n$p(\\lambda|a, \\delta={deltas[delta_idx]:.2f})'+ '= \\sum_{E} p(\\lambda, E | a,' + f'\\delta={deltas[delta_idx]:.2f})$', fontsize=12)
    p_action_params_delta = p_action_params[:, :, :, delta_idx]
    p_action_params_delta = p_action_params_delta / jnp.sum(p_action_params_delta, axis=(1, 2), keepdims=True)
    p_action_params_delta = p_action_params_delta.sum(1) # Marginalize over endowment
    lambdas_means_obs_delta = p_action_params_delta @ lambdas
    

    sns.heatmap(
        p_action_params_delta.T,
        square=True if K_values == len(actions) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="YlOrBr",
        vmin=p_action_params_delta.min(),
        vmax=p_action_params_delta.max(),
        yticklabels=[f'{lambda_:.2f}' for lambda_ in lambdas],
        xticklabels=[f'{a:.2f}' for a in actions],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\lambda|a)$", size=13, rotation=90)
    ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)

    # Known endowment, unknown delta
    ax = axes[0, 2]
    ax.set_title(f'Observer knows endowment $E$\n$p(\\lambda|a, E=${endowment_true_label}$)'+ '= \\sum_{\\delta} p(\\lambda, \\delta| a,' + f'E=${endowment_true_label}$)$', fontsize=12)
    p_action_params_endow = p_action_params.sum(-1)[:, endowment_true]
    p_action_params_endow = p_action_params_endow / jnp.sum(p_action_params_endow, axis=1, keepdims=True)
    lambdas_means_obs_endow = p_action_params_endow @ lambdas

    sns.heatmap(
        p_action_params_endow.T,
        square=True if K_values == len(actions) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Blues",
        vmin=p_action_params_endow.min(),
        vmax=p_action_params_endow.max(),
        yticklabels=[f'{lambda_:.2f}' for lambda_ in lambdas],
        xticklabels=[f'{a:.2f}' for a in actions],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\lambda|a)$", size=13, rotation=90)
    #ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)

    # Kown undewment, known delta
    ax = axes[1, 2]
    ax.set_title(f'Observer knows Endowment $E$ & $\\delta$\n$p(\\lambda|a, E=${endowment_true_label}$, \\delta={deltas[delta_idx]:.2f})$', fontsize=12)
    p_action_params_endow_delta = p_action_params[:, endowment_true, :, delta_idx]
    p_action_params_endow_delta = p_action_params_endow_delta / jnp.sum(p_action_params_endow_delta, axis=1, keepdims=True)
    lambdas_means_obs_endow_delta = p_action_params_endow_delta @ lambdas

    sns.heatmap(
        p_action_params_endow_delta.T,
        square=True if K_values == len(actions) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Purples",
        vmin=p_action_params_endow_delta.min(),
        vmax=p_action_params_endow_delta.max(),
        yticklabels=[f'{lambda_:.2f}' for lambda_ in lambdas],
        xticklabels=[f'{a:.2f}' for a in actions],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\lambda|a)$", size=13, rotation=90)
    ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    #ax.set_yticklabels([], rotation=45, ha='right')

    
    

    ax = axes[0, 0]
    #ax.sharex(axes[1])
    ax.sharex(axes[0, 0])
    ax.sharey(axes[0, 0])

    ax.set_title(f"Bob's naive inferences about Alice's WTR ($\\lambda$) \n $\\delta=0$", size=12, rotation=0)
    p_action_params = Alice_gives_Bob_naive(inv_temp, endowment_belief)
    p_action_params = p_action_params / jnp.sum(p_action_params, axis=(1, 2), keepdims=True)
    p_action_params = p_action_params.sum(1) # Marginalize over endowment
    betas_means_bob_naive = p_action_params @ lambdas

    sns.heatmap(
        p_action_params.T,
        square=True if K_values == len(actions) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Reds",
        vmin=p_action_params.min(),
        vmax=p_action_params.max(),
        yticklabels=[f'{lambda_:.2f}' for lambda_ in lambdas],
        xticklabels=[f'{a:.2f}' for a in actions],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$p(\\lambda|a)$", size=13, rotation=90)
    ax.set_xlabel("Alice's action", size=13)
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    #ax.set_yticklabels([], rotation=45, ha='right')

    # Hide axis [1, 2]
    axes[1, 3].axis('off')
    

    # Create colorbar on the right side without affecting subplot sizes
    #divider = make_axes_locatable(axes[-1])
    #cax = divider.append_axes("right", size="7%", pad=0.0)
    #plt.colorbar(im.collections[0], cax=cax, label='Posterior Probability')

    ax = axes[0, 3]
    ax.sharex(axes[0, 0])
    ax.set_title(f"Probability of each action given WTR ($\\lambda$)\n$p(a|\\lambda, E={endowment_true_label}, \\delta = {deltas[delta_idx]:.2f})$", size=12)
    p_action_params_delta = Alice_gives_Bob(inv_temp, endowment_belief, pres_target, pres_sd)[:, endowment_true, :, delta_idx]
    p_action_params_delta = p_action_params_delta / jnp.sum(p_action_params_delta, axis=0, keepdims=True)

    im = sns.heatmap(
        p_action_params_delta.T,
        square=True if K_values == len(actions) else False,
        annot=True if K_values <= 5 else False,
        fmt=".2f",
        cmap="Greys",
        vmin=p_action_params_delta.min(),
        vmax=p_action_params_delta.max(),
        yticklabels=[f'{lambda_:.2f}' for lambda_ in lambdas],
        xticklabels=[f'{a:.2f}' for a in actions],
        cbar=False,
        ax=ax
    )
    ax.set_ylabel("$\\lambda$", size=13)
    
    ax.tick_params(axis='y', labelsize=10, labelrotation=0)
    ax.tick_params(axis='x', labelsize=10, labelrotation=0)
    ax.set_xlabel("Alice's action", size=13)


    ax = axes[1, 0]
    #ax.sharex(axes[0, 0])
    divider = make_axes_locatable(ax)
    ax1 = divider.append_axes("bottom", size="20%", pad=.4)
    ax.set_title(f"Posterior means WTR $\\lambda$ given each action", size=12)
    

    # Plot the posterior means for each action
    df_means = pd.DataFrame({
        'Action': actions.tolist(),
        'Posterior Mean WTR Lambda Bob Naive': betas_means_bob_naive.tolist(),
        'Posterior Mean WTR Lambda': lambdas_means_obs.tolist(),
        'Posterior Mean WTR Lambda (known delta)': lambdas_means_obs_delta.tolist(),
        'Posterior Mean WTR Lambda (known E)': lambdas_means_obs_endow.tolist(),
        'Posterior Mean WTR Lambda (known E, delta)': lambdas_means_obs_endow_delta.tolist(),
    })
    df_means = df_means.melt(id_vars='Action', value_vars=df_means.columns[1:], var_name='Type', value_name='Mean Lambda')
    palette = sns.color_palette("Reds", n_colors=1) + sns.color_palette("Greens", n_colors=1) + sns.color_palette("YlOrBr", n_colors=1) + sns.color_palette("Blues", n_colors=1) + sns.color_palette("Purples", n_colors=1)
    ax.axhline(y=1, color='gray', linestyle='--')
    sns.barplot(x='Action', y='Mean Lambda', hue='Type', data=df_means, ax=ax, palette=palette)
    
    ax.set_ylabel("$E[\\lambda|a]$", size=13)
    # Put legend at the bottom of the plot
    # Get the handles and labels from the current axes
    handles, labels = ax.get_legend_handles_labels()
    new_labels = ['Naive Bob', 'Observer', 'Observer (knows $\\delta$)', 'Observer (knows E)', 'Observer (knows $E$, $\\delta$)']
    # Remove the legend from this plot to place it elsewhere
    ax.legend().remove()
    #ax.legend(handles, new_labels, loc='center left', bbox_to_anchor=(1.25, 0.5), ncol=1, fontsize=11, borderaxespad=.05)
    sns.despine(ax=ax)

    ax1.axis('off')  # Hide the unused subplot
    # Put the legend here or additional information
    ax1.legend(handles, new_labels, loc='center left', bbox_to_anchor=(-0.25, 0.), ncol=2, fontsize=9, borderaxespad=.05)

    fig.suptitle(f"Inferences about Alice's WTF ($\\lambda$) for Bob and the Observer under different parameters & epistemic states\n Endowment: {endowment_true.name} = {get_endowment_value(endowment_true)}, Bob's belief:" + " $p(\\text{Poor})$" + f" = {endowment_belief:.2f},"  f" $\\rho(\\lambda) = N({pres_target}, {pres_sd})$\n", size=14)
    plt.tight_layout(pad=.005)
    plt.show()

widgets.interactive(
    show_interactive,
    inv_temp=action_inv_temp_widget,
    endowment_true=endowment_widget,
    delta=delta_widget,
    endowment_belief=endowment_belief_widget,
    pres_target=pres_target_widget,
    pres_sd=pres_sd_widget,
)

interactive(children=(FloatSlider(value=4.0, description="Alice's action noise (inv_temp)", layout=Layout(widt…


# Model Predictions

For each prior over the endowment, whether the actual endowment is known or unknown, and for each possible amount received by the receiver, we compute model predictions.



In [13]:
# Create custom widgets that display both value and name
# Create custom widgets with sufficient width for descriptions
def create_labeled_dropdown(options, description, value=None):
    # Convert options to display both value and name
    labeled_options = [(f"{option.name} ({option.value})", option) for option in options]
    
    # Create dropdown with style to accommodate longer descriptions
    dropdown = widgets.Dropdown(
        options=labeled_options, 
        description=description,
        value=value if value is not None else labeled_options[0][1],
        style={'description_width': 'auto'}  # This allows description to use needed space
    )
    
    # Set layout to provide more space
    dropdown.layout.width = '50%'  # Use full width
    
    return dropdown

# Create your custom widgets
#endowment_widget = widgets.FloatSlider(min=1, max=20, step=1, value=20, description="Alice's true endowment", style={'description_width': 'auto'})
endowment_widget = create_labeled_dropdown(Endowment, "Alice's true endowment", value=Endowment.Rich)
action_inv_temp_widget = widgets.FloatSlider(min=0, max=10, step=.2, value=4, description="Alice's action noise (inv_temp)", style={'description_width': 'auto'})
delta_widget = widgets.FloatSlider(min=0, max=1, step=0.1, value=1, description="Alice's care about what Bob thinks: delta", style={'description_width': 'auto'})
#endowment_poor_widget = widgets.FloatSlider(min=1, max=20, step=1, value=1, description="- Bob's Belief of Alice's endowment (Poor)", style={'description_width': 'auto'})
#endowment_rich_widget = widgets.FloatSlider(min=1, max=20, step=1, value=1, description="- Bob's Belief of Alice's endowment (Rich)", style={'description_width': 'auto'})
endowment_belief_widget = widgets.FloatSlider(min=.01, max=.99, step=0.01, value=0.9, description="Bob's Belief that Alice is Rich", style={'description_width': 'auto'})
#endowment_sd_widget = widgets.FloatSlider(min=0.2, max=5, step=.2, value=2, description="- Bob's Belief SD", style={'description_width': 'auto'})
pres_target_widget = widgets.FloatSlider(min=0, max=2, step=0.1, value=1.5, description="Alice's target for Bob's inference of her welfare tradeoff ratio - lambda =", style={'description_width': 'auto'})
pres_sd_widget = widgets.FloatSlider(min=0.05, max=1, step=.05, value=0.25, description="- Target SD", style={'description_width': 'auto'})

action_inv_temp_widget.layout.width = '80%'
endowment_widget.layout.width = '80%'
delta_widget.layout.width = '80%'
endowment_belief_widget.layout.width = '80%'
pres_target_widget.layout.width = '80%'
pres_sd_widget.layout.width = '80%'

def show_interactive(
        inv_temp,
        #endowment_true,
        delta,
        endowment_belief,
        pres_target,
        pres_sd,
    ):

    alpha = alphas[-1]
    delta_idx = jnp.argmin(jnp.abs(deltas - delta))

    poor_idx = actions[actions <= endowments_vals[0]].size
    
    fig = plt.figure(figsize=(16, 16))

    # Add main title for the entire figure
    fig.suptitle('Main Title for Entire Figure', fontsize=16, fontweight='bold', y=1)
    
    # Main grid: 2 rows, 2 columns
    main_gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3, 
                       top=.9, bottom=0.0, left=0.0, right=1)
    
    #actions = jnp.array([.1, .25, .5, .75, .9])
    endowments_priors = jnp.array([endowment_belief, 1-endowment_belief])
    a_v = 100
    observed_bools = jnp.array([False, True])

    for i, endowment_prior in enumerate(endowments_priors):
        for j, observed in enumerate(observed_bools):

            if observed:

                sub_gs = main_gs[i, j].subgridspec(2, 2, wspace=0.3, hspace=0.3)
                sub_axes = [[fig.add_subplot(sub_gs[0, 0]), fig.add_subplot(sub_gs[0, 1])],
                            [fig.add_subplot(sub_gs[1, 0]), fig.add_subplot(sub_gs[1, 1])]]
                
                # Make them all share x and y axes
                for ax_row in sub_axes:
                    for ax in ax_row:
                        ax.sharex(sub_axes[0][0])
                        ax.sharey(sub_axes[0][0])
                
                for k, E in enumerate(Endowment):

                    if k == 0:
                        # Add cell title (spans both subplots)
                        sub_axes[k][0].text(1.15, 1.25, f'Prior={Endowment(i).name} - Observed={observed}', transform=sub_axes[k][0].transAxes,
                             fontsize=15, fontweight='bold', ha='center')

                    ax_wtr = sub_axes[k][0]
                    ax_pres = sub_axes[k][1]

                    p_action_params = Alice_gives_Bob(inv_temp, endowment_prior, pres_target, pres_sd)[:, E, :, :]

                    if E == Endowment.Poor:
                        p_action_params = p_action_params.at[poor_idx:].set(0)  # Actions that are impossible given the endowment
                        p_action_params = p_action_params.at[:poor_idx].set(
                            p_action_params[:poor_idx] / jnp.sum(p_action_params[:poor_idx], axis=(-2, -1), keepdims=True) # Normalize
                        )
                    else:
                        p_action_params = p_action_params / jnp.sum(p_action_params, axis=(-2, -1), keepdims=True) # Normalize

                    # Do WTR
                    p_actions_WTR = p_action_params.sum(-1)  # Marginalize over 
                    sns.heatmap(
                        p_actions_WTR.T,
                        square=True,
                        annot=True if K_values <= 5 else False,
                        fmt=".2f",
                        cmap="Greens",
                        vmin=p_actions_WTR.min(),
                        vmax=p_actions_WTR.max(),
                        yticklabels=[f'{lambda_:.2f}' for lambda_ in lambdas],
                        xticklabels=[f'{a_v*a:.0f}' for a in actions],
                        cbar=False,
                        ax=ax_wtr
                    )
                    Xs = ax_wtr.get_xticks()[:poor_idx] if E == Endowment.Poor else ax_wtr.get_xticks()
                    # Add the mean and sd of the posterior judgements WTR for each action as errorbars
                    Ys_mean = p_actions_WTR @ ax_wtr.get_yticks()
                    Ys_mean = Ys_mean[:poor_idx] if E == Endowment.Poor else Ys_mean
                    ax_wtr.errorbar(x=Xs, y=Ys_mean, fmt='o', color='black', capsize=5)
                    # Add the mode as a triangle
                    Ys_mode = jnp.array([lambdas[jnp.argmax(p_actions_WTR[a_idx])] for a_idx in range(len(actions))])
                    Ys_mode = Ys_mode[:poor_idx] if E == Endowment.Poor else Ys_mode
                    ax_wtr.scatter(x=Xs, y=1/2 + Ys_mode, marker='^', color='blue', label='Mode', zorder=5)

                    ax_wtr.set_ylabel("$p(\\lambda|a)$", size=13, rotation=90)
                    if k == 1:
                        ax_wtr.set_xlabel("Alice's action", size=13)
                    ax_wtr.set_title(f"{E.name} is observed \n Welfare Tradeoff Ratio (WTR)", size=15)

                    ax_wtr.tick_params(axis='y', labelsize=12, labelrotation=0)
                    ax_wtr.tick_params(axis='x', labelsize=12, labelrotation=0)

                    # Do Presentational goal
                    p_actions_PRES = p_action_params.sum(-2)  # Marginalize over lambda
                    sns.heatmap(
                        p_actions_PRES.T,
                        square=True,
                        annot=True if K_values <= 5 else False,
                        fmt=".2f",
                        cmap="Oranges",
                        vmin=p_actions_PRES.min(),
                        vmax=p_actions_PRES.max(),
                        yticklabels=[f'{delta_:.2f}' for delta_ in deltas],
                        xticklabels=[f'{a_v*a:.0f}' for a in actions],
                        cbar=False,
                        ax=ax_pres
                    )
                    # Add the mean and sd of the posterior judgements PRES for each action as errorbars
                    Xs = ax_pres.get_xticks()[:poor_idx] if E == Endowment.Poor else ax_pres.get_xticks()
                    Ys_mean = p_actions_PRES @ ax_pres.get_yticks()
                    Ys_mean = Ys_mean[:poor_idx] if E == Endowment.Poor else Ys_mean
                    ax_pres.errorbar(x=Xs, y=Ys_mean, fmt='o', color='black', capsize=5) 
                    # Add the mode as a triangle
                    Ys_mode = jnp.array([deltas[jnp.argmax(p_actions_PRES[a_idx])] for a_idx in range(len(actions))])
                    Ys_mode = Ys_mode[:poor_idx] if E == Endowment.Poor else Ys_mode
                    ax_pres.scatter(x=Xs, y=1/2 + Ys_mode, marker='^', color='blue', label='Mode', zorder=5)

                    ax_pres.set_title(f"{E.name} is observed\n Presentational Goal ($\\delta$)", size=15)
                    if k == 1:
                        ax_pres.set_xlabel("Alice's action", size=13)
                    ax_pres.set_ylabel("$p(\\delta|a)$", size=13, rotation=90)
                    ax_pres.tick_params(axis='y', labelsize=12, labelrotation=0)
                    ax_pres.tick_params(axis='x', labelsize=12, labelrotation=0)

            
            else:
                sub_gs = main_gs[i, j].subgridspec(1, 2, wspace=0.3)
                ax_wtr = fig.add_subplot(sub_gs[0])
                ax_pres = fig.add_subplot(sub_gs[1])

                # SHare x and y axes
                ax_wtr.sharex(ax_pres)
                ax_wtr.sharey(ax_pres)

                p_action_params = Alice_gives_Bob(inv_temp, endowment_prior, pres_target, pres_sd).sum(1)  # Marginalize over endowment

                p_action_params = p_action_params / jnp.sum(p_action_params, axis=(-2, -1), keepdims=True) # Normalize

                # Add cell title (spans both subplots)
                ax_wtr.text(1.15, 1.95, f'Prior={Endowment(i).name} - Observed={observed}', transform=ax_wtr.transAxes,
                     fontsize=15, fontweight='bold', ha='center')
    
                # Do WTR
                p_actions_WTR = p_action_params.sum(-1)  # Marginalize over delta
    
                sns.heatmap(
                    p_actions_WTR.T,
                    square=True,
                    annot=True if K_values <= 5 else False,
                    fmt=".2f",
                    cmap="Greens",
                    vmin=p_actions_WTR.min(),
                    vmax=p_actions_WTR.max(),
                    yticklabels=[f'{lambda_:.2f}' for lambda_ in lambdas],
                    xticklabels=[f'{a_v*a:.0f}' for a in actions],
                    cbar=False,
                    ax=ax_wtr
                )
                # Add the mean and sd of the posterior judgements WTR for each action as errorbars
                Xs = ax_wtr.get_xticks()
                Ys_mean = p_actions_WTR @ ax_wtr.get_yticks()
                ax_wtr.errorbar(x=Xs, y=Ys_mean, fmt='o', color='black', capsize=5)
                # Add the mode as a triangle
                Ys_mode = jnp.array([lambdas[jnp.argmax(p_actions_WTR[a_idx])] for a_idx in range(len(actions))])
                ax_wtr.scatter(x=Xs, y=1/2 + Ys_mode, marker='^', color='blue', label='Mode', zorder=5)
                
    
                ax_wtr.set_ylabel("$p(\\lambda|a)$", size=13, rotation=90)
                ax_wtr.set_xlabel("Alice's action", size=13)
                ax_wtr.set_title("Welfare Tradeoff Ratio (WTR)", size=15)
        
                ax_wtr.tick_params(axis='y', labelsize=12, labelrotation=0)
                ax_wtr.tick_params(axis='x', labelsize=12, labelrotation=0)
    
                # Do Presentational goal
                p_actions_PRES = p_action_params.sum(-2)  # Marginalize over lambda
    
                sns.heatmap(
                    p_actions_PRES.T,
                    square=True,
                    annot=True if K_values <= 5 else False,
                    fmt=".2f",
                    cmap="Oranges",
                    vmin=p_actions_PRES.min(),
                    vmax=p_actions_PRES.max(),
                    yticklabels=[f'{delta_:.2f}' for delta_ in deltas],
                    xticklabels=[f'{a_v*a:.0f}' for a in actions],
                    cbar=False,
                    ax=ax_pres
                )
                # Add the mean and sd of the posterior judgements PRES for each action as errorbars
                ax_pres.errorbar(x=ax_pres.get_xticks(), y=p_actions_PRES @ ax_pres.get_yticks(), fmt='o', color='black', capsize=5) 
                # Add the mode as a triangle
                ax_pres.scatter(x=ax_pres.get_xticks(), y=1/2 + jnp.array([deltas[jnp.argmax(p_actions_PRES[a_idx])] for a_idx in range(len(actions))]), marker='^', color='blue', label='Mode', zorder=5)
                ax_pres.set_title("Presentational Goal ($\\delta$)", size=15)
                ax_pres.set_xlabel("Alice's action", size=13)
                ax_pres.set_ylabel("$p(\\delta|a)$", size=13, rotation=90)
    
                ax_pres.tick_params(axis='y', labelsize=12, labelrotation=0)
                ax_pres.tick_params(axis='x', labelsize=12, labelrotation=0)

    
    # Add a legend for the mean and mode under the first column
    legend_ax = fig.add_subplot(main_gs[1, 0])
    legend_ax.axis('off')
    legend_ax.scatter([], [], marker='o', color='black', label='Posterior Mean', zorder=5)
    legend_ax.scatter([], [], marker='^', color='blue', label='Posterior Mode', zorder=5)
    # Make the legend at the bottom center of the figure
    legend_ax.legend(loc='center', fontsize=15, borderaxespad=0.5, frameon=True, bbox_to_anchor=(0.5, 0), ncol=2)
    

    fig.suptitle(f"Inferences about Alice's WTF ($\\lambda$) for Bob and the Observer under different parameters & epistemic states\nBob's belief:" + " $p(\\text{Poor})$" + f" = {endowment_belief:.2f},"  f" $\\rho(\\lambda) = N({pres_target}, {pres_sd})$\n", size=15, y=1, fontweight='bold')
    plt.show()

widgets.interactive(
    show_interactive,
    inv_temp=action_inv_temp_widget,
    #endowment_true=endowment_widget,
    delta=delta_widget,
    endowment_belief=endowment_belief_widget,
    pres_target=pres_target_widget,
    pres_sd=pres_sd_widget,
)

interactive(children=(FloatSlider(value=4.0, description="Alice's action noise (inv_temp)", layout=Layout(widt…